# Pathogenicity Ensemble

**Predicting whether a protein variant is pathogenic by folding it.**

Given a wild-type (WT) protein sequence and its mutated (MUT) counterpart, this
notebook predicts whether the variant is *pathogenic* or *benign* — not from
sequence alone, but by actually predicting the 3D structure of both versions
and learning from how the fold changed.

---

## The idea

A sequence-only classifier sees `...R414C...` and has to infer everything from
context. But the reason a missense variant is damaging is usually structural:
it buries a charge, breaks a salt bridge, disrupts a binding interface, or
loosens a domain. So here both versions of the protein are folded with
**OpenFold3**, superimposed, and the *difference between the two structures*
becomes the model's input.

Two problems have to be solved to make that practical:

1. **Proteins are too big to fold.** APC is ~2,843 residues — far beyond a
   single GPU. Solved by folding only a **±250-residue window centered on the
   mutation** (501 residues), cut identically from the WT and MUT sequences.
2. **Two independent folds live in unrelated reference frames.** Solved with a
   **Kabsch superposition** before any coordinates are compared.

## Pipeline

```
   CSV of variants (WT + MUT full-length sequences, ClinVar labels)
                              │
                              ▼
          ±250-residue window centered on the mutation
                              │
            ┌─────────────────┴─────────────────┐
            ▼                                   ▼
      WT window                            MUT window
            │  OpenFold3                        │  OpenFold3
            ▼                                   ▼
      WT structure (.cif)                 MUT structure (.cif)
            └─────────────────┬─────────────────┘
                              ▼
                   Kabsch superposition
                              │
        ┌─────────────────────┼─────────────────────┐
        ▼                     ▼                     ▼
  ESM2 embeddings      16 structural          residue graph of
 (full sequences)       descriptors        the WT→MUT difference
        │                     │                     │
    ESM2_MLP            STRUCT_MLP                 GNN
        └─────────────────────┼─────────────────────┘
                              ▼
            logistic-regression meta-learner (stacking)
                     under nested cross-validation
                              ▼
                     pathogenic / benign
```

Three models are used because they fail differently: the sequence branch sees
the whole protein but no geometry, the descriptor branch sees global shape
change, and the GNN sees local structural rewiring. The meta-learner learns how
much to trust each one.

## Dataset

`protein_mutation_dataset_APC.csv` — **51 ClinVar-derived APC variants**
(the tumour-suppressor gene behind familial adenomatous polyposis), with
full-length WT and mutated sequences, HGVS notation, ClinVar allele/variation
IDs, and a pathogenic/benign label. The file is **not committed to this repo**;
place it next to the notebook before running.

## Runtime

Built for **Google Colab with an A100 40GB GPU** and Drive mounted. Folding all
102 windows is the expensive step (hours) and is checkpointed to disk, so the
modeling cell at the end can be re-run freely without re-folding anything.

> See [`README.md`](README.md) for the same overview outside the notebook.

---

## 1 · Environment

Install the folding stack and download the OpenFold3 weights. `setup_openfold`
is interactive but safe to re-run — it detects existing parameters and skips
the multi-gigabyte download.

In [ ]:
# openfold3[cuequivariance] : structure prediction + fused equivariant kernels
# biopython / gemmi         : sequence and mmCIF structure parsing
# py3Dmol                   : in-notebook 3D structure rendering
#
# Requires a GPU runtime. The pipeline below was developed on a Colab A100 40GB;
# folding a 501-residue window needs roughly 20-30GB with the low-memory preset.
!pip install -q "openfold3[cuequivariance]" biopython pandas gemmi py3Dmol

In [ ]:
# One-time interactive setup: downloads the OpenFold3 weights
# (openbind-2025-06-30-174k, ~several GB) into /root/.openfold3 and syncs the
# Biotite CCD chemical component dictionary used to build ligand/residue templates.
# Safe to re-run - it detects existing parameters and skips the download.
!setup_openfold

Setting up OpenFold3...
Please specify the OpenFold cache directory (default: /root/.openfold3): 
Please specify the directory for parameter download (default: /root/.openfold3): 
Select parameters to download:
1) Download only the default checkpoint (openbind-2025-06-30-174k)
2) Download all parameters (openbind-2025-06-30-174k)
3) Download a specific parameter by name
Enter your choice (1/2/3, default: 1): 
Force re-download parameters even if they already exist? (yes/no, default: no) 
Run integration tests? (yes/no) 
Parameters directory set to: /root/.openfold3
Starting parameter download...
Parameters already present at /root/.openfold3/of3-ob-2025-06-30-174k.pt
Download completed successfully.
Starting Biotite CCD setup...
Biotite CCD file at /usr/local/lib/python3.13/dist-packages/biotite/structure/info/components.bcif is up-to-date with s3://openfold3-data/components.bcif, skipping.
Skipping integration tests.
Setup configuration saved to /root/.openfold3/setup_config.json


## 2 · Load the variant dataset

One row per variant. The row index becomes the **sample id** used in every
downstream filename (`sample_<i>_WT_window.json`, `sample_<i>_MUT_best.cif`),
which is why `reset_index` matters here.

In [ ]:
# protein_mutation_dataset_APC.csv holds ClinVar-derived APC variants, one row
# per variant, with the columns:
#   gene, transcript_accession, protein_accession   - identifiers
#   transcript_hgvs, protein_hgvs                   - e.g. c.1240C>T / p.Arg414Cys
#   normal_sequence, mutated_sequence               - full-length WT and MUT protein
#   label                                           - 'pathogenic' or 'benign'
#   allele_id, variation_id                         - ClinVar cross-references
#
# NOTE: this CSV is not committed to the repo; place it beside the notebook.
import os
import json
import pandas as pd
from pathlib import Path

CSV_PATH = "protein_mutation_dataset_APC.csv"

# dropna() discards rows where sequence reconstruction failed upstream.
# reset_index matters: the row index becomes the sample id used in every
# downstream filename (sample_<i>_WT_window.json, sample_<i>_MUT_best.cif, ...).
df = pd.read_csv(CSV_PATH).dropna().reset_index(drop=True)

# Normalize a protein sequence: strip whitespace, uppercase, and drop the '*'
# stop codon marker so that nonsense variants yield a plain residue string.
def clean_seq(seq):
    return str(seq).strip().upper().replace("*", "")

# openfold_queries/ holds one JSON query per folding job;
# openfold_results/ receives one subdirectory of predicted mmCIF models per job.
INPUT_DIR = Path("openfold_queries")
RESULT_DIR = Path("openfold_results")

INPUT_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

print("Samples:", len(df))
display(df.head())

Samples: 51


,gene,transcript_accession,protein_accession,transcript_hgvs,protein_hgvs,normal_sequence,mutated_sequence,label,source,status,allele_id,variation_id
0,APC,NM_000038.6,NP_000029.2,c.123A>G,p.Ala41=,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,benign,protein_hgvs,ok,617040,631238
1,APC,NM_000038.6,NP_000029.2,c.1240C>T,p.Arg414Cys,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,benign,protein_hgvs,ok,15836,797
2,APC,NM_000038.6,NP_000029.2,c.1240del,p.Arg414fs,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,pathogenic,cdna_hgvs,ok,432574,438864
3,APC,NM_000038.6,NP_000029.2,c.1242C>T,p.Arg414=,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,benign,protein_hgvs,ok,212403,215553
4,APC,NM_000038.6,NP_000029.2,c.1333C>T,p.Gln445Ter,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,MAAASYDQLLKQVEALKMENSNLRQELEDNSNHLTKLETEASNMKE...,pathogenic,protein_hgvs,ok,432575,438865


## 3 · CUDA / CUTLASS toolchain

OpenFold3 JIT-compiles fused equivariant kernels against CUTLASS at import
time, so these environment variables must be set **before** the first heavy
import in the folding cells below.

The allocator settings are not cosmetic: folding long windows fragments GPU
memory badly, and without `expandable_segments` the run OOMs partway through
the batch. Drive is mounted so predicted structures survive a runtime
disconnect.

In [ ]:
# CUTLASS supplies the CUDA templates that the cuEquivariance triangle-attention
# kernels are JIT-compiled against. Installed separately so CUTLASS_PATH below
# can point at a version that matches the runtime's CUDA toolkit.
!pip install -U nvidia-cutlass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 38.0 MB/s eta 0:00:00


In [ ]:
# OpenFold3 compiles fused kernels at import time, so these must be set *before*
# the first heavy import in the folding cells.
import os
import pathlib
import cutlass_library
import torch

os.environ["CUDA_HOME"] = "/usr/local/cuda"

# Derive CUTLASS_PATH from the installed wheel rather than hardcoding a path.
os.environ["CUTLASS_PATH"] = str(
    pathlib.Path(cutlass_library.__file__).resolve().parent / "source"
)

os.environ["LD_LIBRARY_PATH"] = (
    f"{os.environ['CUDA_HOME']}/lib64:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

# Folding long windows fragments GPU memory badly. expandable_segments lets the
# caching allocator grow a segment instead of reserving a new one, and capping
# split size keeps large activation tensors from stranding free blocks.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:128"
)

# Load CUDA modules on first use to cut start-up time and idle VRAM.
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

torch.cuda.empty_cache()

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

!nvidia-smi

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Sun Sep 13 08:12:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |


In [ ]:
# Mount Drive to persist predicted structures between Colab sessions - folding
# all 102 windows takes hours, so the outputs should outlive the runtime.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4 · Low-memory folding profile

Written to `cueq_gpu_runner.yml` and passed to every `run_openfold predict`
call via `--runner-yaml`. **Without these settings a 501-residue window does
not fit on a 40GB A100.**

The trade is speed for memory across the board: minimum chunk size, low-memory
attention instead of the faster fused triangle kernels, and the MSA module and
confidence heads offloaded to CPU whenever they are idle.

In [ ]:
%%writefile cueq_gpu_runner.yml
# ============================================================
# Low-memory inference profile for OpenFold3
# ============================================================
# Written to disk and passed to every `run_openfold predict` call via
# --runner-yaml. Without these settings a 501-residue window OOMs on a 40GB A100.
model_update:
  presets:
    # 'predict' = inference-only preset; 'low_mem' trades speed for VRAM headroom.
    - "predict"
    - "low_mem"

  custom:
    settings:
      memory:
        eval:
          # Process attention in the smallest possible chunks (slowest, least memory).
          chunk_size: 1

          # Kernel selection: the fused triangle-attention kernels are faster but need
          # more working memory, so fall back to low-memory attention (LMA) instead.
          use_cueq_triangle_kernels: false
          use_deepspeed_evo_attention: false
          use_lma: true
          use_triton_triangle_kernels: false

          # Offload the MSA module and confidence heads to CPU between uses.
          # token_cutoff: 0 means 'always offload', regardless of sequence length.
          offload_inference:
            msa_module: true
            confidence_heads: true
            token_cutoff: 0

          # Hard ceilings per sample. These bound the work for any single query and are
          # what the window radius of 250 below is sized against (501 = 2*250 + 1).
          per_sample_token_cutoff: 512
          per_sample_atom_cutoff: 4096

Writing cueq_gpu_runner.yml


## 5 · The windowing trick

This is the step that makes the whole project feasible.

APC is ~2,843 residues. Instead of folding it, locate the mutation and cut an
**identical window from both sequences**:

```
WT   ────────────────────[ ······· D ······· ]────────────────────
MUT  ────────────────────[ ······· N ······· ]────────────────────
                         ↑         ↑         ↑
                    center-250   center   center+250
```

Both windows are 501 residues and cover exactly the same positions, so the two
predicted structures are directly comparable residue-by-residue.

**Caveat, stated up front:** the window is centered on the *first differing
residue*. For missense variants that is exactly right. For frameshift and
nonsense variants every residue downstream changes, so a fixed window captures
only part of the effect — this dataset contains such variants (`p.Arg414fs`,
`p.Gln445Ter`), and the global ESM2 branch later is what partly compensates.

This cell runs the logic on one sample and writes two OpenFold3 query JSONs.

In [ ]:
# APC is ~2,843 residues - far beyond what fits on one GPU. The key idea of this
# project is to fold only a window centered on the mutation, for both the WT and
# the MUT sequence, and learn from the *difference* between the two structures.
# This cell validates that on sample 0 before the full batch run.
import json
import pandas as pd
from pathlib import Path

CSV_PATH = "protein_mutation_dataset_APC.csv"
df = pd.read_csv(CSV_PATH).dropna().reset_index(drop=True)

def clean_seq(seq):
    return str(seq).strip().upper().replace("*", "")

# Locate the mutation and cut an identical window from both sequences.
#
# Returns the two aligned windows plus the mutation's position in global
# (`center`) and window-local (`local_center`) coordinates.
#
# CAVEAT: only the FIRST differing residue is used as the center. For missense
# variants that is exactly right. For frameshift/nonsense variants every residue
# downstream changes, so a fixed window captures only part of the effect.
def find_mutation_window(wt_seq, mut_seq, radius=250):
    wt_seq = clean_seq(wt_seq)
    mut_seq = clean_seq(mut_seq)

    n = min(len(wt_seq), len(mut_seq))

    # Positionwise comparison over the overlapping prefix. Indels shift the
    # register, so for those this flags the first shifted position, not a
    # substituted residue.
    mutation_positions = [
        i for i in range(n)
        if wt_seq[i] != mut_seq[i]
    ]

    # Synonymous variants (e.g. p.Ala41=) have identical protein sequences;
    # fall back to the middle of the protein so the sample still folds.
    if len(mutation_positions) == 0:
        center = n // 2
    else:
        center = mutation_positions[0]

    # Clamp to the sequence bounds - mutations near either terminus produce a
    # shorter-than-501 window rather than an out-of-range slice.
    start = max(0, center - radius)
    end = min(n, center + radius + 1)

    return {
        "wt_window": wt_seq[start:end],
        "mut_window": mut_seq[start:end],
        "start": start,
        "end": end,
        "center": center,
        "local_center": center - start,
    }

SAMPLE_INDEX = 0
WINDOW_RADIUS = 250

row = df.loc[SAMPLE_INDEX]

window_data = find_mutation_window(
    row["normal_sequence"],
    row["mutated_sequence"],
    radius=WINDOW_RADIUS
)

print("Original protein length:", len(clean_seq(row["normal_sequence"])))
print("Window length:", len(window_data["wt_window"]))
print("Mutation global position:", window_data["center"])
print("Mutation local position:", window_data["local_center"])
print("WT residue:", window_data["wt_window"][window_data["local_center"]])
print("MUT residue:", window_data["mut_window"][window_data["local_center"]])

INPUT_DIR = Path("openfold_queries")
INPUT_DIR.mkdir(exist_ok=True)

# Build one OpenFold3 query per variant. The schema is:
#   {"queries": {<name>: {"chains": [{molecule_type, chain_ids, sequence}]}}}
# Single protein chain A, no ligands, no templates - monomer folding only.
queries = {
    "sample_0_WT_window": window_data["wt_window"],
    "sample_0_MUT_window": window_data["mut_window"],
}

for name, seq in queries.items():
    query = {
        "queries": {
            name: {
                "chains": [
                    {
                        "molecule_type": "protein",
                        "chain_ids": ["A"],
                        "sequence": seq
                    }
                ]
            }
        }
    }

    out_json = INPUT_DIR / f"{name}.json"

    with open(out_json, "w") as f:
        json.dump(query, f, indent=2)

    print("Created:", out_json)

Original protein length: 2843
Window length: 501
Mutation global position: 1421
Mutation local position: 250
WT residue: D
MUT residue: D
Created: openfold_queries/sample_0_WT_window.json
Created: openfold_queries/sample_0_MUT_window.json


## 6 · Fold the first sample

A dry run on sample 0 before committing GPU-hours to all 102 jobs. Each call
fetches MSAs from the ColabFold server (so this needs network access) and
writes predicted models to:

```
openfold_results/<name>/<name>/seed_42/<name>_seed_42_sample_1_model.cif
```

The cell after it re-reads the query JSON to confirm the 501-residue window
made it through intact.

In [ ]:
# Each call writes predicted mmCIF models under
#   openfold_results/<name>/<name>/seed_42/..._sample_1_model.cif
# MSAs are fetched from the ColabFold MSA server, so this step needs network
# access and takes several minutes per window.
import torch
# Release cached blocks so the folding subprocess starts with a clean GPU.
torch.cuda.empty_cache()

for name in ["sample_0_WT_window", "sample_0_MUT_window"]:
    print("Running:", name)

    # Clear any partial output from an interrupted run before re-folding.
    !rm -rf openfold_results/{name}
    !mkdir -p openfold_results/{name}

    !run_openfold predict \
    --query-json=openfold_queries/{name}.json \
    --output-dir=openfold_results/{name} \
    --runner-yaml=cueq_gpu_runner.yml

Running: sample_0_WT_window
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Submitting 1 sequences to the Colabfold MSA server for main MSAs...
COMPLETE: 100% 150/150 [00:47<00:00,  3.13it/s]
/usr/local/lib/python3.13/dist-packages/openfold3/core/data/tools/colabfold_msa_server.py:401: DeprecationWarning: Python 3.14 will, by default, filter

In [ ]:
# Sanity check: confirm the query JSON holds the expected 501-residue window
# before committing GPU hours to the full batch.
import json

with open("openfold_queries/sample_0_WT_window.json") as f:
    q = json.load(f)

seq = q["queries"]["sample_0_WT_window"]["chains"][0]["sequence"]

print("Sequence length:", len(seq))
print(seq[:50], "...", seq[-50:])

Sequence length: 501
HVDQPIDYSLKYATDIPSSQKQSFSFSKSSSGQSSKTEHMSSSSENTSTP ... RLQPQKHVSFTPGDDMPRVYCVEGTPINFSTATSLSDLTIESPPNELAAG


## 7 · Comparing the two structures

The WT and MUT windows were folded independently, so their coordinates are in
arbitrary, unrelated reference frames. Subtracting them directly would measure
nothing but that arbitrariness.

**Kabsch alignment** finds the rotation that minimises RMSD between the two
sets of alpha carbons:

1. center both point sets on their centroids
2. take the SVD of the covariance matrix `H = P₀ᵀQ₀`
3. the optimal rotation is `R = VUᵀ` — with a determinant check, because a
   negative determinant means the "rotation" is really a reflection

The transform is derived from alpha carbons only, then replayed onto every
atom. What comes out is **per-residue displacement** — the core structural
signal this project is built on — plus a `py3Dmol` overlay of the WT (gray)
against the mutation-replaced geometry (red).

For sample 0 the displacements turn out to be tiny (mean ≈ 0.05 Å), which is
the expected result: sample 0 is the synonymous variant `p.Ala41=`, where the
protein sequence does not change at all. A good negative control.

In [ ]:
# The two windows are folded independently, so their coordinates live in
# arbitrary, unrelated reference frames. Before any structural difference is
# meaningful they must be superimposed - that is what the Kabsch step does.
!pip install -q gemmi py3Dmol

from pathlib import Path
import numpy as np
import pandas as pd
import gemmi
import py3Dmol
from IPython.display import display, HTML

# OpenFold3 emits several sampled models per seed; take the first
# deterministically so WT and MUT are compared consistently.
def first_cif(folder):
    cifs = sorted(Path(folder).rglob("*.cif"))
    if len(cifs) == 0:
        raise FileNotFoundError(f"No CIF files found in {folder}")
    return cifs[0]

wt_cif = first_cif("openfold_results/sample_0_WT_window")
mut_cif = first_cif("openfold_results/sample_0_MUT_window")

print("WT CIF:", wt_cif)
print("MUT CIF:", mut_cif)

# Flatten an mmCIF into a tidy atom table (one row per atom) for easy
# filtering, alignment and export.
def load_atom_df(cif_path):
    structure = gemmi.read_structure(str(cif_path))
    rows = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    rows.append({
                        "chain": chain.name,
                        "residue_id": residue.seqid.num,
                        "residue_name": residue.name,
                        "atom_name": atom.name,
                        "x": float(atom.pos.x),
                        "y": float(atom.pos.y),
                        "z": float(atom.pos.z),
                    })

    return pd.DataFrame(rows)

wt_df = load_atom_df(wt_cif)
mut_df = load_atom_df(mut_cif)

# Alpha carbons only: one representative point per residue, which is the
# standard backbone-level abstraction for fold comparison.
def get_ca(df):
    ca = df[df["atom_name"] == "CA"].copy()
    ca = ca.sort_values("residue_id").reset_index(drop=True)
    return ca[["x", "y", "z"]].values

# Kabsch algorithm - the rotation minimizing RMSD between two point sets.
#   1. center both sets on their centroids
#   2. SVD of the covariance matrix H = P0^T Q0
#   3. R = V U^T is the optimal rotation
# Returns the aligned points plus (R, source centroid, target centroid) so the
# same transform can be replayed onto every atom, not just the CAs.
def kabsch_align(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)

    Pc = P.mean(axis=0)
    Qc = Q.mean(axis=0)

    P0 = P - Pc
    Q0 = Q - Qc

    H = P0.T @ Q0
    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T

    # Guard against a reflection: a negative determinant means the 'rotation'
    # mirrors the structure. Flipping the last singular vector restores a proper
    # rotation (det = +1).
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    P_aligned = P0 @ R + Qc

    return P_aligned, R, Pc, Qc

wt_ca = get_ca(wt_df)
mut_ca = get_ca(mut_df)

# Indels can make the two windows differ in length; compare the common prefix.
n = min(len(wt_ca), len(mut_ca))

mut_ca_aligned, R, mut_center, wt_center = kabsch_align(
    mut_ca[:n],
    wt_ca[:n]
)

# Replay the CA-derived transform onto the full atom table.
def apply_alignment(df, R, source_center, target_center):
    coords = df[["x", "y", "z"]].values
    aligned = (coords - source_center) @ R + target_center

    out = df.copy()
    out["x"] = aligned[:, 0]
    out["y"] = aligned[:, 1]
    out["z"] = aligned[:, 2]
    return out

mut_aligned_df = apply_alignment(mut_df, R, mut_center, wt_center)

# This is the mutation-replaced local geometry.
# Since we are using a local window, the replaced structure is the aligned MUT window.
hybrid_df = mut_aligned_df.copy()

wt_df.to_csv("sample_0_WT_window_coordinates.csv", index=False)
mut_aligned_df.to_csv("sample_0_MUT_window_aligned_coordinates.csv", index=False)
hybrid_df.to_csv("sample_0_mutation_replaced_window_coordinates.csv", index=False)

print("Saved coordinate CSVs.")
print("WT atoms:", wt_df.shape)
print("MUT aligned atoms:", mut_aligned_df.shape)

# displacement analysis
# Per-residue displacement after superposition - the core structural signal.
# Small values mean the mutation barely perturbs the backbone; large or
# locally concentrated values suggest a disruptive substitution.
disp = np.sqrt(np.sum((mut_ca_aligned[:n] - wt_ca[:n]) ** 2, axis=1))

print("Mean CA displacement:", float(np.mean(disp)))
print("Max CA displacement:", float(np.max(disp)))
print("Mutation local residue index:", window_data["local_center"])

# Render WT and mutation-replaced geometry together
# Serialize the atom table to fixed-column PDB text for py3Dmol. The column
# offsets below are the PDB ATOM record spec, not arbitrary formatting.
def df_to_pdb(df, title="MODEL"):
    lines = []
    atom_id = 1

    for _, r in df.iterrows():
        lines.append(
            f"ATOM  {atom_id:5d} {r['atom_name']:<4} {r['residue_name']:>3} {r['chain']:1}"
            f"{int(r['residue_id']):4d}    "
            f"{r['x']:8.3f}{r['y']:8.3f}{r['z']:8.3f}"
            f"  1.00  0.00           {r['atom_name'][0]:>2}"
        )
        atom_id += 1

    lines.append("END")
    return "\n".join(lines)

wt_pdb = df_to_pdb(wt_df)
hybrid_pdb = df_to_pdb(hybrid_df)

# Overlay both structures: WT in gray, mutation-replaced geometry in red.
viewer = py3Dmol.view(width=1000, height=750)

viewer.addModel(wt_pdb, "pdb")
viewer.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})

viewer.addModel(hybrid_pdb, "pdb")
viewer.setStyle({"model": 1}, {"cartoon": {"color": "red"}})

viewer.zoomTo()
display(HTML(viewer._make_html()))

WT CIF: openfold_results/sample_0_WT_window/sample_0_WT_window/seed_42/sample_0_WT_window_seed_42_sample_1_model.cif
MUT CIF: openfold_results/sample_0_MUT_window/sample_0_MUT_window/seed_42/sample_0_MUT_window_seed_42_sample_1_model.cif
Saved coordinate CSVs.
WT atoms: (3772, 7)
MUT aligned atoms: (3772, 7)
Mean CA displacement: 0.052328553367015966
Max CA displacement: 0.17101802329437593
Mutation local residue index: 250


## 8 · Scale to the full dataset

Same windowing logic applied to every row: **2 queries per sample (WT + MUT) =
102 folding jobs**.

The batch runner is written to be **idempotent and resumable**, because Colab
runtimes disconnect and this stage takes hours: a query that already has CIF
output is skipped rather than re-folded, and a failure is reported and stepped
over so one bad sample cannot abort the rest.

Everything is then zipped, which cleanly decouples the two halves of the
project — the expensive GPU stage runs once, and the model below can be
retrained from the archive any number of times.

In [ ]:
# Same windowing logic as the dry run, applied to every row: 2 queries per
# sample (WT + MUT) = 102 folding jobs for the 51-variant APC dataset.
import json
import pandas as pd
from pathlib import Path

CSV_PATH = "protein_mutation_dataset_APC.csv"

df = pd.read_csv(CSV_PATH).dropna().reset_index(drop=True)

def clean_seq(seq):
    return str(seq).strip().upper().replace("*", "")

def find_mutation_window(wt_seq, mut_seq, radius=250):

    wt_seq = clean_seq(wt_seq)
    mut_seq = clean_seq(mut_seq)

    n = min(len(wt_seq), len(mut_seq))

    mutation_positions = [
        i for i in range(n)
        if wt_seq[i] != mut_seq[i]
    ]

    if len(mutation_positions) == 0:
        center = n // 2
    else:
        center = mutation_positions[0]

    start = max(0, center - radius)
    end = min(n, center + radius + 1)

    return {
        "wt_window": wt_seq[start:end],
        "mut_window": mut_seq[start:end],
        "start": start,
        "end": end,
        "center": center,
    }

INPUT_DIR = Path("openfold_queries")
RESULT_DIR = Path("openfold_results")

INPUT_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

WINDOW_RADIUS = 250

# Manifest of (sample index, variant, query name, json path) tuples, consumed
# by the batch folding cell below.
query_files = []

for i, row in df.iterrows():

    window_data = find_mutation_window(
        row["normal_sequence"],
        row["mutated_sequence"],
        radius=WINDOW_RADIUS
    )

    queries = {
        f"sample_{i}_WT_window": window_data["wt_window"],
        f"sample_{i}_MUT_window": window_data["mut_window"],
    }

    for query_name, seq in queries.items():

        query = {
            "queries": {
                query_name: {
                    "chains": [
                        {
                            "molecule_type": "protein",
                            "chain_ids": ["A"],
                            "sequence": seq
                        }
                    ]
                }
            }
        }

        out_json = INPUT_DIR / f"{query_name}.json"

        with open(out_json, "w") as f:
            json.dump(query, f, indent=2)

        variant = "WT" if "WT" in query_name else "MUT"

        query_files.append(
            (i, variant, query_name, out_json)
        )

print("Total window queries:", len(query_files))
print(query_files[:4])

Total window queries: 102
[(0, 'WT', 'sample_0_WT_window', PosixPath('openfold_queries/sample_0_WT_window.json')), (0, 'MUT', 'sample_0_MUT_window', PosixPath('openfold_queries/sample_0_MUT_window.json')), (1, 'WT', 'sample_1_WT_window', PosixPath('openfold_queries/sample_1_WT_window.json')), (1, 'MUT', 'sample_1_MUT_window', PosixPath('openfold_queries/sample_1_MUT_window.json'))]


In [ ]:
# Run OpenFold only on local WT/MUT mutation windows
# Long-running (hours). Written to be idempotent and resumable: Colab runtimes
# disconnect, so a sample that already has CIF output is skipped rather than
# re-folded. Failures are reported and skipped so one bad sample cannot abort
# the remaining jobs.
import os

for sample_i, variant, query_name, query_json in query_files:
    out_dir = RESULT_DIR / query_name

    # Presence of a CIF is the completion marker for this query.
    existing_cifs = list(out_dir.rglob("*.cif")) if out_dir.exists() else []

    if existing_cifs:
        print("Skipping existing:", query_name)
        continue

    print("Running:", query_name)

    out_dir.mkdir(exist_ok=True)

    cmd = f"""
    run_openfold predict \
    --query-json={query_json} \
    --output-dir={out_dir} \
    --runner-yaml=cueq_gpu_runner.yml
    """

    result = os.system(cmd)

    if result != 0:
        print("FAILED:", query_name)
    else:
        print("DONE:", query_name)

Skipping existing: sample_0_WT_window
Skipping existing: sample_0_MUT_window
Running: sample_1_WT_window
DONE: sample_1_WT_window
Running: sample_1_MUT_window
DONE: sample_1_MUT_window
Running: sample_2_WT_window


In [ ]:
# Bundle every predicted structure into a single zip. The modeling cell below
# consumes this archive, which keeps folding and training fully decoupled: the
# expensive GPU stage runs once, the model can then be retrained from the zip.
from pathlib import Path
import shutil
from google.colab import files

RESULT_DIR = Path("openfold_results")

zip_name = "all_openfold_predictions"

# Create zip
shutil.make_archive(
    zip_name,
    'zip',
    RESULT_DIR
)

zip_path = f"{zip_name}.zip"

print("Created zip:", zip_path)

# Download automatically
files.download(zip_path)

---

# 9 · Features, models and the stacked ensemble

Everything below consumes the folded structures and trains the classifier. It
is one self-contained cell: feature construction, three base models, and the
stacking layer.

## Features extracted per variant

| Source | What it captures | Shape |
| --- | --- | --- |
| **Global ESM2** | mean-pooled embeddings of the **full-length** WT and MUT sequences, plus their signed and absolute difference | 4 blocks concatenated |
| **Structural descriptors** | RMSD, displacement statistics (mean/std/max/p90/at-the-mutation/local), radius of gyration for WT, MUT and Δ, contact gains and losses | 16 values |
| **Residue graph** | one node per residue; features = per-residue ESM2 delta ‖ WT one-hot ‖ displacement ‖ coordinate delta ‖ distance-to-mutation ‖ two masks | ~500 nodes |

## The graph is the interesting part

A single graph carries **both structures at once**. An edge exists between two
residues if they are in contact (≤ 8 Å between alpha carbons) in *either*
structure — so contacts **created** by the mutation are represented just as
much as contacts **destroyed**. Each edge carries five attributes:

```
[ WT distance | MUT distance | Δ distance | contact_changed | backbone? ]
```

`contact_changed` is the XOR across the cutoff: 1 only when a contact appeared
or vanished. This matters because a variant can wreck a binding interface while
barely moving the backbone — contact rewiring often carries more signal than
raw displacement.

## The three branches

| Branch | View | Model |
| --- | --- | --- |
| `ESM2_MLP` | sequence, whole protein | MLP over PCA-reduced ESM2 features |
| `STRUCT_MLP` | structure, global | tiny MLP over the 16 interpretable descriptors |
| `GNN` | structure, local | 2× `GINEConv` + four pooled readouts |

`GINEConv` is the edge-feature-aware variant of GIN — necessary here, since the
signal lives on the edges. Readout mixes mean and max pooling over the whole
graph with **masked pooling over the mutation site and its ±10-residue
neighborhood**, so local effects are not diluted across 500 residues.

## Why nested cross-validation

The meta-learner is trained on base-model predictions. If those predictions
came from models that had already seen the same rows, they would look far more
confident than they really are and the stack would be badly miscalibrated. So:

```
outer fold (10×)  ──►  scores the ensemble honestly
    └── inner fold (3×)  ──►  produces out-of-fold predictions
                              that train the meta-learner
```

Every preprocessing step (`StandardScaler`, `PCA`) is likewise fit on training
rows only. A fixed 0.80 / 0.10 / 0.10 blend is evaluated alongside the learned
stack as a baseline: **if the learned stack cannot beat it, the structural
branches are not earning their folding compute.**

With only ~51 samples, all of this is guarding against a very real risk of
optimistic results.

In [ ]:
# ============================================================
# ESM2 + OpenFold GNN + Structure Features + Stacked Ensemble
# ============================================================

!pip install -q fair-esm gemmi torch-geometric scikit-learn pandas biopython

import os, zipfile, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import gemmi
import torch
import torch.nn as nn
import torch.nn.functional as F
import esm

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
# GINEConv is the edge-feature-aware variant of GIN. It is used here because the
# edge attributes carry the signal - distances in the WT vs MUT structure and
# whether a contact appeared or vanished.
from torch_geometric.nn import GINEConv, global_mean_pool, global_max_pool

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, log_loss, f1_score

# ============================================================
# Settings
# ============================================================

CSV_PATH = "protein_mutation_dataset_APC.csv"
ZIP_PATH = "all_openfold_predictions.zip"

EXTRACT_DIR = Path("openfold_unzipped")
DEDUP_DIR = Path("openfold_deduped")
CACHE_DIR = Path("gnn_cache")

GRAPH_CACHE_PATH = CACHE_DIR / "graph_data_with_struct.pt"
ESM_CACHE_PATH = CACHE_DIR / "esm_global_features_with_struct.npz"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Single seed threaded through numpy/torch/sklearn for reproducibility.
# N_SPLITS  = outer folds, used to score the ensemble.
# INNER_SPLITS = inner folds, used only to produce meta-features for stacking.
SEED = 42
N_SPLITS = 10
INNER_SPLITS = 3

# The ESM2 feature vector is far wider than the number of samples, so it is
# compressed with PCA before the MLP to avoid a degenerate fit.
# DIST_CUTOFF (8 Angstrom between alpha carbons) is the standard definition of a
# residue-residue contact; it sets both the graph edges and the contact maps.
PCA_COMPONENTS = 32
DIST_CUTOFF = 8.0

GNN_EPOCHS = 40
MLP_EPOCHS = 120
STRUCT_EPOCHS = 120

# Small batches: each graph has ~500 nodes and a dense contact-derived edge set,
# so a handful of graphs already fills GPU memory.
BATCH_SIZE = 4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CACHE_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)

# ============================================================
# Load CSV
# ============================================================

df = pd.read_csv(CSV_PATH).dropna().reset_index(drop=True)

def clean_seq(seq):
    return str(seq).strip().upper().replace("*", "")

# Binarize the ClinVar-style label. Anything not explicitly (likely) pathogenic
# - benign, likely benign, VUS - collapses to the negative class.
def label_to_int(v):
    text = str(v).strip().lower()
    if text in {"1", "pathogenic", "likely pathogenic"}:
        return 1
    return 0

df["normal_sequence"] = df["normal_sequence"].map(clean_seq)
df["mutated_sequence"] = df["mutated_sequence"].map(clean_seq)
df["y"] = df["label"].map(label_to_int)

y = df["y"].values.astype(int)

print("Samples:", len(df))
print("Pathogenic:", int(y.sum()))
print("Non-pathogenic:", int(len(y) - y.sum()))

# ============================================================
# Extract ZIP only if needed
# ============================================================

# Unzip only when no structures are present, so re-running the cell is cheap.
if not EXTRACT_DIR.exists() or len(list(EXTRACT_DIR.rglob("*.cif"))) == 0:
    print("Extracting ZIP...")

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    EXTRACT_DIR.mkdir(exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
else:
    print("Using existing extracted folder:", EXTRACT_DIR)

print("Extracted CIF count:", len(list(EXTRACT_DIR.rglob("*.cif"))))

# ============================================================
# Deduplicate CIFs
# ============================================================

# Locate every predicted model for one (sample, variant). Two glob patterns
# cover both the '_window' naming used by the batch run and the plain naming
# from earlier ad-hoc runs.
def find_all_cifs(sample_i, variant):
    patterns = [
        f"**/sample_{sample_i}_{variant}_window/**/*.cif",
        f"**/sample_{sample_i}_{variant}/**/*.cif",
    ]

    hits = []
    for p in patterns:
        hits.extend(list(EXTRACT_DIR.glob(p)))

    return sorted(set(hits))

# Deterministic single-model choice per sample/variant.
#
# CAVEAT: 'best' here is lexicographically first, NOT highest confidence.
# OpenFold3 emits per-model confidence (pLDDT/PAE) that is currently unused -
# ranking by it, or averaging several models, is an obvious improvement.
def pick_best_cif(cif_list):
    if len(cif_list) == 0:
        return None
    return sorted(cif_list)[0]

if not DEDUP_DIR.exists() or len(list(DEDUP_DIR.rglob("*.cif"))) < 2:
    print("Creating deduplicated CIF folder...")

    if DEDUP_DIR.exists():
        shutil.rmtree(DEDUP_DIR)

    DEDUP_DIR.mkdir(exist_ok=True)

    kept = 0

    for i in range(len(df)):
        for variant in ["WT", "MUT"]:
            cifs = find_all_cifs(i, variant)
            best = pick_best_cif(cifs)

            if best is None:
                continue

            out_name = f"sample_{i}_{variant}_best.cif"
            shutil.copy2(best, DEDUP_DIR / out_name)
            kept += 1

    print("Deduplicated CIFs kept:", kept)
else:
    print("Using existing deduplicated CIF folder:", DEDUP_DIR)

# Keep only samples where BOTH the WT and the MUT window folded successfully -
# every feature below is a difference, so an unpaired structure is useless.
pairs = []

for i in range(len(df)):
    wt_cif = DEDUP_DIR / f"sample_{i}_WT_best.cif"
    mut_cif = DEDUP_DIR / f"sample_{i}_MUT_best.cif"

    if wt_cif.exists() and mut_cif.exists():
        pairs.append((i, wt_cif, mut_cif))

print("Complete WT/MUT structure pairs:", len(pairs))

if len(pairs) == 0:
    raise RuntimeError("No WT/MUT CIF pairs found.")

# ============================================================
# Structure utilities
# ============================================================

# ---- Structure utilities ----
# mmCIF stores 3-letter residue names; ESM2 and the one-hot encoder want the
# 1-letter alphabet.
AA3_TO_1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D",
    "CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
    "HIS":"H","ILE":"I","LEU":"L","LYS":"K",
    "MET":"M","PHE":"F","PRO":"P","SER":"S",
    "THR":"T","TRP":"W","TYR":"Y","VAL":"V",
}

# Canonical 20 amino acids, fixed order so the one-hot encoding is stable
# across samples and runs.
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {a:i for i,a in enumerate(AA_ORDER)}

# Unknown/non-standard residues yield an all-zero vector rather than an error.
def one_hot_aa(a):
    out = np.zeros(len(AA_ORDER), dtype=np.float32)
    if a in AA_TO_IDX:
        out[AA_TO_IDX[a]] = 1.0
    return out

# Read one predicted structure down to a per-residue alpha-carbon table.
# Sorting by residue_id guarantees WT and MUT rows correspond position-by-position.
def load_ca_from_cif(cif_path):
    structure = gemmi.read_structure(str(cif_path))
    rows = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.name == "CA":
                        rows.append({
                            "residue_id": residue.seqid.num,
                            "residue_name": residue.name,
                            "aa": AA3_TO_1.get(residue.name.upper(), "X"),
                            "x": float(atom.pos.x),
                            "y": float(atom.pos.y),
                            "z": float(atom.pos.z),
                        })

    ca = pd.DataFrame(rows)
    ca = ca.sort_values("residue_id").reset_index(drop=True)
    return ca

def kabsch_align(P, Q):
    P = np.asarray(P, dtype=np.float32)
    Q = np.asarray(Q, dtype=np.float32)

    Pc = P.mean(axis=0)
    Qc = Q.mean(axis=0)

    P0 = P - Pc
    Q0 = Q - Qc

    H = P0.T @ Q0
    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1,:] *= -1
        R = Vt.T @ U.T

    return P0 @ R + Qc

# Same first-difference logic as the windowing step, but applied to the sequences
# read back OUT of the predicted structures - which is what the node indices
# actually correspond to.
def find_first_mutation_position(wt_seq, mut_seq):
    n = min(len(wt_seq), len(mut_seq))

    for i in range(n):
        if wt_seq[i] != mut_seq[i]:
            return i

    return n // 2

# Overall compactness of the fold. A mutation that unfolds or loosens a domain
# shows up as an increase relative to the WT.
def radius_of_gyration(coords):
    coords = np.asarray(coords, dtype=np.float32)
    center = coords.mean(axis=0)
    return float(np.sqrt(np.mean(np.sum((coords - center) ** 2, axis=1))))

# Symmetric binary residue-residue contact matrix. O(n^2) in residues, which is
# acceptable at ~500 residues per window.
def contact_map(coords, cutoff=8.0):
    coords = np.asarray(coords, dtype=np.float32)
    n = len(coords)
    cm = np.zeros((n, n), dtype=np.float32)

    # Spatial (non-local) contact edges over all residue pairs.
    for i in range(n):
        diff = coords[i + 1:] - coords[i]
        dists = np.sqrt(np.sum(diff ** 2, axis=1))
        hits = np.where(dists <= cutoff)[0]

        for h in hits:
            j = i + 1 + h
            cm[i, j] = 1.0
            cm[j, i] = 1.0

    return cm

# Collapse a WT/MUT structure pair into 16 fixed descriptors - the input to the
# STRUCT_MLP branch. These are deliberately interpretable: unlike the GNN, every
# column here has a physical meaning.
def compute_structure_tabular_features(wt_xyz, mut_aligned, mut_pos, cutoff=8.0):
    n = min(len(wt_xyz), len(mut_aligned))

    wt_xyz = wt_xyz[:n]
    mut_aligned = mut_aligned[:n]

    # Per-residue backbone displacement after superposition.
    disp = np.sqrt(np.sum((mut_aligned - wt_xyz) ** 2, axis=1))

    wt_rg = radius_of_gyration(wt_xyz)
    mut_rg = radius_of_gyration(mut_aligned)

    wt_cm = contact_map(wt_xyz, cutoff=cutoff)
    mut_cm = contact_map(mut_aligned, cutoff=cutoff)

    # 1 wherever a contact was gained or lost. Contact rewiring is often a better
    # pathogenicity signal than raw displacement, since a variant can disrupt a
    # binding interface while barely moving the backbone.
    contact_diff = np.abs(mut_cm - wt_cm)

    total_possible = max(n * (n - 1), 1)

    contact_changed_count = float(contact_diff.sum() / 2.0)
    contact_changed_fraction = float(contact_diff.sum() / total_possible)

    # Guard the index: truncation above can push the mutation site past the end.
    mut_pos = int(np.clip(mut_pos, 0, n - 1))
    # A +/-10 residue neighborhood around the mutation - the region most likely to
    # be perturbed, isolated so local effects are not diluted by the whole window.
    local_radius = 10

    local_start = max(0, mut_pos - local_radius)
    local_end = min(n, mut_pos + local_radius + 1)

    local_disp = disp[local_start:local_end]
    local_cm_diff = contact_diff[local_start:local_end, :]

    feats = [
        float(n),  # window length in residues
        float(np.sqrt(np.mean(disp ** 2))),  # global RMSD (WT vs MUT)
        float(np.mean(disp)),  # mean displacement
        float(np.std(disp)),  # displacement spread
        float(np.max(disp)),  # worst-moved residue
        float(np.percentile(disp, 90)),  # tail of the displacement distribution
        float(disp[mut_pos]),  # displacement AT the mutated residue
        float(np.mean(local_disp)),  # mean displacement near the mutation
        float(np.max(local_disp)),  # max displacement near the mutation
        float(wt_rg),  # WT compactness
        float(mut_rg),  # MUT compactness
        float(mut_rg - wt_rg),  # change in compactness (+ = looser fold)
        contact_changed_count,  # number of contacts gained/lost
        contact_changed_fraction,  # same, normalized by window size
        float(local_cm_diff.sum()),  # contact changes involving the local region
        float(local_cm_diff.mean()),  # same, per residue pair
    ]

    return np.array(feats, dtype=np.float32)

# Build the graph edges. This is the core structural idea of the project:
# a single graph carries BOTH structures at once. Each edge knows how far apart
# its two residues are in the WT, how far apart in the MUT, and whether the
# contact between them appeared or disappeared.
#
# An edge exists if the pair is in contact in EITHER structure, so contacts
# created by the mutation are represented just as much as contacts destroyed.
def build_mutation_aware_edges(wt_coords, mut_coords, cutoff=8.0):
    wt_coords = np.asarray(wt_coords, dtype=np.float32)
    mut_coords = np.asarray(mut_coords, dtype=np.float32)

    n = min(len(wt_coords), len(mut_coords))

    edges = []
    edge_attrs = []

    # Each edge carries 5 attributes:
    #   [0] WT distance      (scaled by 20A into roughly [0,1])
    #   [1] MUT distance     (same scaling)
    #   [2] signed change in distance
    #   [3] contact_changed  - 1 if the pair crossed the cutoff either way
    #   [4] edge_type        - 1 = backbone bond, 0 = spatial contact
    def add_edge(i, j, wt_dist, mut_dist, edge_type):
        # XOR across the cutoff: true only when the contact was gained or lost.
        contact_changed = float((wt_dist <= cutoff) != (mut_dist <= cutoff))

        edge_attrs.append([
            wt_dist / 20.0,
            mut_dist / 20.0,
            (mut_dist - wt_dist) / 20.0,
            contact_changed,
            edge_type,
        ])

        edges.append([i, j])

    # Sequential backbone edges (i <-> i+1), added in both directions because
    # PyTorch Geometric treats edge_index as directed.
    for i in range(n - 1):
        wt_d = np.linalg.norm(wt_coords[i] - wt_coords[i + 1])
        mut_d = np.linalg.norm(mut_coords[i] - mut_coords[i + 1])

        add_edge(i, i + 1, wt_d, mut_d, 1.0)
        add_edge(i + 1, i, wt_d, mut_d, 1.0)

    for i in range(n):
        for j in range(i + 1, n):
            wt_d = np.linalg.norm(wt_coords[i] - wt_coords[j])
            mut_d = np.linalg.norm(mut_coords[i] - mut_coords[j])

            if wt_d <= cutoff or mut_d <= cutoff:
                add_edge(i, j, wt_d, mut_d, 0.0)
                add_edge(j, i, wt_d, mut_d, 0.0)

    # Degenerate fallback so a pathological structure still yields a valid graph.
    if len(edges) == 0:
        edges = [[0,0]]
        edge_attrs = [[0,0,0,0,0]]

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float32)

    return edge_index, edge_attr

# ============================================================
# Load ESM2
# ============================================================

print("Loading ESM2...")

# ---- ESM2 protein language model ----
# The smallest ESM2 checkpoint (6 transformer layers). Chosen because it runs
# twice per sample plus once per residue window on a single GPU; the larger
# checkpoints would likely improve the sequence branch at a real compute cost.
esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
esm_model = esm_model.eval().to(DEVICE)

@torch.no_grad()
# Per-residue embeddings, one vector per amino acid.
def esm_residue_embeddings(seq):
    seq = clean_seq(seq)
    data = [("protein", seq)]

    _, _, tokens = batch_converter(data)
    tokens = tokens.to(DEVICE)

    out = esm_model(tokens, repr_layers=[6], return_contacts=False)
    # Take layer 6 (the final layer) and slice off the BOS/EOS special tokens so
    # row i lines up with residue i of the input sequence.
    rep = out["representations"][6][0, 1:len(seq)+1]

    return rep.detach().cpu().numpy().astype(np.float32)

@torch.no_grad()
# Whole-protein summary: mean-pool the residue embeddings into one vector.
def esm_mean_embedding(seq):
    emb = esm_residue_embeddings(seq)
    return emb.mean(axis=0).astype(np.float32)

# ============================================================
# Build/load graph + ESM + structure feature cache
# ============================================================

# ---- Feature construction (cached) ----
# Building graphs runs ESM2 over every window and computes O(n^2) contact maps,
# so results are cached to disk and reused on subsequent runs. Delete gnn_cache/
# to force a rebuild after changing any feature definition.
if GRAPH_CACHE_PATH.exists() and ESM_CACHE_PATH.exists():
    print("Loading cached graph + ESM + structure features...")

    cached = torch.load(GRAPH_CACHE_PATH, weights_only=False)

    graph_data = cached["graph_data"]
    valid_indices = cached["valid_indices"]

    esm_cache = np.load(ESM_CACHE_PATH)

    X_esm_global = esm_cache["X_esm_global"]
    X_struct = esm_cache["X_struct"]
    y_valid = esm_cache["y_valid"]

else:
    print("Building graph dataset...")

    graph_data = []
    global_esm_features = []
    structure_features = []
    valid_indices = []

    # Main feature-building loop: one WT/MUT structure pair -> one graph,
    # one structural descriptor vector, and one global ESM2 vector.
    for sample_i, wt_cif, mut_cif in pairs:
        print("Building graph sample:", sample_i)

        wt_ca = load_ca_from_cif(wt_cif)
        mut_ca = load_ca_from_cif(mut_cif)

        n = min(len(wt_ca), len(mut_ca))

        if n < 3:
            continue

        wt_ca = wt_ca.iloc[:n].reset_index(drop=True)
        mut_ca = mut_ca.iloc[:n].reset_index(drop=True)

        wt_xyz = wt_ca[["x","y","z"]].values.astype(np.float32)
        mut_xyz = mut_ca[["x","y","z"]].values.astype(np.float32)

        # Superimpose MUT onto WT. Without this the coordinate differences below would
        # just measure two arbitrary reference frames, not a conformational change.
        mut_aligned = kabsch_align(mut_xyz, wt_xyz)

        displacement = np.sqrt(
            np.sum((mut_aligned - wt_xyz) ** 2, axis=1)
        ).reshape(-1,1)

        # Recover the sequence from the structure itself, so residue indices, embeddings
        # and coordinates are guaranteed to refer to the same positions.
        # Unresolved residues ('X') are mapped to alanine so ESM2 accepts the string.
        wt_local_seq = "".join(wt_ca["aa"].tolist()).replace("X","A")
        mut_local_seq = "".join(mut_ca["aa"].tolist()).replace("X","A")

        wt_res_esm = esm_residue_embeddings(wt_local_seq)
        mut_res_esm = esm_residue_embeddings(mut_local_seq)

        # Truncate every array to one common length before stacking them into nodes.
        m = min(n, wt_res_esm.shape[0], mut_res_esm.shape[0])

        wt_xyz = wt_xyz[:m]
        mut_aligned = mut_aligned[:m]
        displacement = displacement[:m]

        wt_res_esm = wt_res_esm[:m]
        mut_res_esm = mut_res_esm[:m]

        aa_features = np.vstack([
            one_hot_aa(a)
            for a in wt_local_seq[:m]
        ])

        # Signed 3D shift per residue - direction of motion, where `displacement` above
        # only captures magnitude.
        coord_delta = mut_aligned - wt_xyz

        mut_pos = find_first_mutation_position(
            wt_local_seq,
            mut_local_seq
        )

        # Positional encoding relative to the mutation site: the model should weight
        # residues near the variant more heavily than distant ones.
        pos = np.arange(m, dtype=np.float32)

        dist_from_mut = np.abs(pos - mut_pos).reshape(-1,1)
        dist_from_mut_norm = dist_from_mut / max(m,1)

        mutation_mask = (dist_from_mut == 0).astype(np.float32)
        near_mut_mask = (dist_from_mut <= 10).astype(np.float32)

        # Node feature vector, concatenated per residue:
        #   ESM2 delta   (D dims) - how the language model's view of this residue changed
        #   aa one-hot   (20)     - WT identity
        #   displacement (1)      - how far it moved
        #   coord_delta  (3)      - which way it moved
        #   dist_from_mut(1)      - normalized sequence distance to the variant
        #   masks        (2)      - exactly at / within 10 residues of the variant
        #
        # ORDER MATTERS: StructureGNN.forward reads the last three columns back out
        # by index, so these three must stay last and in this order.
        node_features = np.concatenate([
            mut_res_esm - wt_res_esm,
            aa_features,
            displacement,
            coord_delta,
            dist_from_mut_norm,
            mutation_mask,
            near_mut_mask,
        ], axis=1).astype(np.float32)

        # Edges come from BOTH structures (see build_mutation_aware_edges).
        edge_index, edge_attr = build_mutation_aware_edges(
            wt_xyz,
            mut_aligned,
            cutoff=DIST_CUTOFF
        )

        # One PyTorch Geometric graph per variant. sample_index is retained so a
        # prediction can always be traced back to its ClinVar row.
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor([y[sample_i]], dtype=torch.float32),
            sample_index=sample_i,
            mut_pos=torch.tensor([mut_pos], dtype=torch.long),
        )

        graph_data.append(data)
        valid_indices.append(sample_i)

        struct_feat = compute_structure_tabular_features(
            wt_xyz,
            mut_aligned,
            mut_pos,
            cutoff=DIST_CUTOFF
        )

        structure_features.append(struct_feat)

        # Global ESM2 features use the FULL-LENGTH sequences, not the folded window -
        # this branch can therefore see effects outside the structural window
        # (truncations, frameshifts) that the GNN branch structurally misses.
        wt_global = esm_mean_embedding(df.loc[sample_i, "normal_sequence"])
        mut_global = esm_mean_embedding(df.loc[sample_i, "mutated_sequence"])

        # Four blocks: WT embedding, MUT embedding, their signed difference, and the
        # absolute difference. The difference terms give the classifier the change
        # directly instead of making it learn a subtraction.
        global_feat = np.concatenate([
            wt_global,
            mut_global,
            mut_global - wt_global,
            np.abs(mut_global - wt_global),
        ]).astype(np.float32)

        global_esm_features.append(global_feat)

    valid_indices = np.array(valid_indices)
    y_valid = y[valid_indices]

    X_esm_global = np.vstack(global_esm_features).astype(np.float32)
    X_struct = np.vstack(structure_features).astype(np.float32)

    torch.save({
        "graph_data": graph_data,
        "valid_indices": valid_indices,
    }, GRAPH_CACHE_PATH)

    np.savez_compressed(
        ESM_CACHE_PATH,
        X_esm_global=X_esm_global,
        X_struct=X_struct,
        y_valid=y_valid,
    )

print("Graph samples:", len(graph_data))
print("Global ESM features:", X_esm_global.shape)
print("Structure features:", X_struct.shape)

# ============================================================
# Models
# ============================================================

# ============================================================
# Branch 1 - sequence view: MLP over PCA-reduced ESM2 features
# ============================================================
class ESMMLP(nn.Module):
    def __init__(self, input_dim, hidden=64, dropout=0.25):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# ============================================================
# Branch 2 - global structural view: MLP over the 16 descriptors
# ============================================================
# Deliberately tiny (24 -> 12 -> 1). With 16 inputs and few dozen samples,
# a larger head would memorize the training folds outright.
class StructureFeatureMLP(nn.Module):
    def __init__(self, input_dim, hidden=24, dropout=0.25):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 12),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(12, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# ============================================================
# Branch 3 - local structural view: GNN over the WT/MUT difference graph
# ============================================================
# Two GINEConv layers, so each residue aggregates information from its spatial
# neighbors and their neighbors - roughly a two-hop structural neighborhood.
# Readout deliberately mixes whole-graph and mutation-focused pooling.
class StructureGNN(nn.Module):
    def __init__(self, input_dim, edge_dim=5, hidden=48, dropout=0.45):
        super().__init__()

        self.node_proj = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        def mlp():
            return nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden),
            )

        # GINE propagates node AND edge features, which is what lets the contact-change
        # flag on each edge influence the message passing.
        self.conv1 = GINEConv(mlp(), edge_dim=edge_dim)
        self.conv2 = GINEConv(mlp(), edge_dim=edge_dim)

        self.dropout = dropout

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 4 + 4, 48),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(48,1),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        # Pull the three positional/mask columns out of the RAW node features before
        # they are projected away, so they can be used for masked pooling and as
        # graph-level scalars. These indices mirror the node_features layout above.
        raw_dist = x[:, -3]
        raw_mut_mask = x[:, -2]
        raw_near_mask = x[:, -1]

        x = self.node_proj(x)

        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Four complementary graph readouts:
        #   mean - overall character of the structural change
        #   max  - the single most perturbed residue
        #   near - average over the mutation neighborhood only (masked)
        #   mut  - the mutated residue itself (masked)
        mean_pool = global_mean_pool(x, batch)
        max_pool = global_max_pool(x, batch)

        near_pool = global_mean_pool(x * raw_near_mask.unsqueeze(1), batch)
        mut_pool = global_max_pool(x * raw_mut_mask.unsqueeze(1), batch)

        # Per-graph scalar summaries appended to the pooled vector. The Python loop is
        # acceptable only because BATCH_SIZE is 4; it would be a bottleneck at scale and
        # could be replaced with scatter reductions.
        batch_size = int(batch.max().item()) + 1
        scalar_rows = []

        for b in range(batch_size):
            mask = batch == b

            scalar_rows.append(torch.stack([
                raw_dist[mask].mean(),
                raw_dist[mask].min(),
                raw_near_mask[mask].mean(),
                raw_mut_mask[mask].sum(),
            ]))

        scalar_pool = torch.stack(scalar_rows, dim=0).to(x.device)

        pooled = torch.cat([
            mean_pool,
            max_pool,
            near_pool,
            mut_pool,
            scalar_pool,
        ], dim=1)

        return self.classifier(pooled).squeeze(-1)

# ============================================================
# Training helpers
# ============================================================

# ============================================================
# Training helpers - each returns validation-fold probabilities
# ============================================================
# Full-batch gradient descent: the dataset is small enough to fit on the GPU at
# once, so there is no minibatch loop here.
def train_mlp(X_train, y_train, X_val, epochs=120):
    model = ESMMLP(X_train.shape[1]).to(DEVICE)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=8e-4,
        weight_decay=1e-4
    )

    loss_fn = nn.BCEWithLogitsLoss()

    X_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)

    model.train()

    for _ in range(epochs):
        opt.zero_grad()
        logits = model(X_t)
        loss = loss_fn(logits, y_t)
        loss.backward()
        opt.step()

    model.eval()

    with torch.no_grad():
        X_v = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        prob = torch.sigmoid(model(X_v)).detach().cpu().numpy()

    return np.clip(prob, 1e-6, 1 - 1e-6)

# Same shape as train_mlp, with three stabilizers for the tiny tabular input:
# logit clamping, label smoothing, and gradient-norm clipping.
def train_struct_mlp(X_train, y_train, X_val, epochs=120):
    model = StructureFeatureMLP(X_train.shape[1]).to(DEVICE)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4,
        weight_decay=5e-4
    )

    loss_fn = nn.BCEWithLogitsLoss()

    X_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)

    model.train()

    for _ in range(epochs):
        opt.zero_grad()

        logits = model(X_t)
        logits = torch.clamp(logits, -5, 5)

        # Label smoothing: targets become 0.05/0.95 instead of 0/1, which stops the
        # model from driving logits to infinity on an easily separable small dataset
        # and keeps the probabilities usable as stacking inputs.
        y_smooth = y_t * 0.9 + 0.05

        loss = loss_fn(logits, y_smooth)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        opt.step()

    model.eval()

    with torch.no_grad():
        X_v = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        prob = torch.sigmoid(model(X_v)).detach().cpu().numpy()

    return np.clip(prob, 1e-6, 1 - 1e-6)

# Heavier regularization than the other branches (dropout 0.45, weight decay
# 2e-3): the GNN has by far the most parameters relative to the sample count.
def train_gnn(train_graphs, val_graphs, input_dim, epochs=40):
    model = StructureGNN(input_dim).to(DEVICE)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=2e-3
    )

    loss_fn = nn.BCEWithLogitsLoss()

    loader = DataLoader(
        train_graphs,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    best_state = None
    best_loss = float("inf")
    patience = 20
    wait = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch in loader:
            batch = batch.to(DEVICE)

            opt.zero_grad()

            logits = model(
                batch.x,
                batch.edge_index,
                batch.edge_attr,
                batch.batch
            )

            logits = torch.clamp(logits, -5, 5)

            y_smooth = batch.y.view(-1) * 0.9 + 0.05

            loss = loss_fn(logits, y_smooth)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            opt.step()

            total_loss += loss.item()

        avg_loss = total_loss / max(len(loader),1)

        # Checkpoint the best epoch.
        #
        # CAVEAT: selection is on TRAINING loss, since the validation fold must stay
        # unseen. This guards against a diverging run rather than against overfitting.
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = {
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    probs = []

    val_loader = DataLoader(
        val_graphs,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)

            prob = torch.sigmoid(
                model(
                    batch.x,
                    batch.edge_index,
                    batch.edge_attr,
                    batch.batch
                )
            )

            probs.extend(prob.detach().cpu().numpy().tolist())

    return np.clip(np.array(probs), 1e-6, 1 - 1e-6)

# ============================================================
# Base prediction helper for stacking
# ============================================================

# ============================================================
# Fit all three branches on one train/val split
# ============================================================
# Every preprocessing step (StandardScaler, PCA) is fit on the TRAINING rows
# only and then applied to the validation rows. Fitting them on the full matrix
# would leak validation information and inflate every metric below.
def get_base_predictions(train_idx, val_idx):
    # ESM2 branch
    esm_scaler = StandardScaler()
    X_esm_tr = esm_scaler.fit_transform(X_esm_global[train_idx])
    X_esm_va = esm_scaler.transform(X_esm_global[val_idx])

    # PCA cannot produce more components than samples-1; inner folds are small,
    # so the requested 32 is clamped rather than raising.
    n_comp = min(PCA_COMPONENTS, X_esm_tr.shape[0] - 1, X_esm_tr.shape[1])
    pca = PCA(n_components=n_comp, random_state=SEED)

    X_esm_tr = pca.fit_transform(X_esm_tr)
    X_esm_va = pca.transform(X_esm_va)

    mlp_prob = train_mlp(
        X_esm_tr,
        y_valid[train_idx],
        X_esm_va,
        epochs=MLP_EPOCHS
    )

    # Structure MLP branch
    struct_scaler = StandardScaler()
    X_struct_tr = struct_scaler.fit_transform(X_struct[train_idx])
    X_struct_va = struct_scaler.transform(X_struct[val_idx])

    struct_prob = train_struct_mlp(
        X_struct_tr,
        y_valid[train_idx],
        X_struct_va,
        epochs=STRUCT_EPOCHS
    )

    # GNN branch
    train_graphs = [graph_data[i] for i in train_idx]
    val_graphs = [graph_data[i] for i in val_idx]

    gnn_input_dim = graph_data[0].x.shape[1]

    gnn_prob = train_gnn(
        train_graphs,
        val_graphs,
        gnn_input_dim,
        epochs=GNN_EPOCHS
    )

    return mlp_prob, gnn_prob, struct_prob

# ============================================================
# Cross-validation with logistic stacking
# ============================================================

# ============================================================
# Nested cross-validation with logistic stacking
# ============================================================
# Outer loop  -> honest performance estimate for the ensemble.
# Inner loop  -> out-of-fold base predictions used to TRAIN the meta-learner.
#
# The nesting is the point: if the meta-learner were fit on predictions from
# models that had already seen those rows, the base models would look far more
# confident than they are and the stack would be badly miscalibrated.
# Stratified splits keep the pathogenic/benign ratio stable in every fold.
outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

# Out-of-fold prediction vectors: every sample is scored exactly once, by a
# model that never trained on it.
mlp_probs_oof = np.zeros(len(y_valid))
gnn_probs_oof = np.zeros(len(y_valid))
struct_probs_oof = np.zeros(len(y_valid))
stacked_probs_oof = np.zeros(len(y_valid))
fixed_probs_oof = np.zeros(len(y_valid))

fold_results = []

for fold, (tr, va) in enumerate(outer_cv.split(X_esm_global, y_valid), 1):
    print(f"\nFold {fold}/{N_SPLITS}")

    tr = np.array(tr)
    va = np.array(va)

    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED + fold
    )

    # Meta-feature matrix for this outer fold: one column per base model.
    meta_train = np.zeros((len(tr), 3))

    for inner_fold, (itr, iva) in enumerate(inner_cv.split(X_esm_global[tr], y_valid[tr]), 1):
        inner_train_idx = tr[itr]
        inner_val_idx = tr[iva]

        print(f"  Inner fold {inner_fold}/{INNER_SPLITS}")

        inner_mlp, inner_gnn, inner_struct = get_base_predictions(
            inner_train_idx,
            inner_val_idx
        )

        meta_train[iva, 0] = inner_mlp
        meta_train[iva, 1] = inner_gnn
        meta_train[iva, 2] = inner_struct

    # The meta-learner. Intentionally a linear model with strong regularization
    # (C=0.5) over just 3 inputs - it learns how much to trust each branch, and its
    # coefficients are directly readable. class_weight balances the label skew.
    meta_model = LogisticRegression(
        solver="liblinear",
        C=0.5,
        class_weight="balanced",
        random_state=SEED
    )

    meta_model.fit(meta_train, y_valid[tr])

    # Now refit the base models on the FULL outer-training set and predict the
    # held-out fold, then push those through the fitted meta-model.
    mlp_prob, gnn_prob, struct_prob = get_base_predictions(tr, va)

    meta_val = np.column_stack([
        mlp_prob,
        gnn_prob,
        struct_prob
    ])

    stacked_prob = meta_model.predict_proba(meta_val)[:, 1]

    # Baseline for comparison: a hand-set blend heavily weighted toward the sequence
    # branch. If the learned stack cannot beat this, the structural branches are not
    # contributing and the extra folding compute is not paying for itself.
    fixed_prob = (
        0.80 * mlp_prob
        + 0.10 * gnn_prob
        + 0.10 * struct_prob
    )

    mlp_probs_oof[va] = mlp_prob
    gnn_probs_oof[va] = gnn_prob
    struct_probs_oof[va] = struct_prob
    stacked_probs_oof[va] = stacked_prob
    fixed_probs_oof[va] = fixed_prob

    for name, prob in [
        ("ESM2_MLP", mlp_prob),
        ("GNN", gnn_prob),
        ("STRUCT_MLP", struct_prob),
        ("FIXED_ENSEMBLE", fixed_prob),
        ("STACKED_ENSEMBLE", stacked_prob),
    ]:
        pred = (prob >= 0.5).astype(int)

        acc = accuracy_score(y_valid[va], pred)
        auc = roc_auc_score(y_valid[va], prob)
        ap = average_precision_score(y_valid[va], prob)
        loss = log_loss(y_valid[va], prob, labels=[0,1])
        f1 = f1_score(y_valid[va], pred, zero_division=0)

        print(
            name,
            "acc:", round(acc,3),
            "auc:", round(auc,3),
            "ap:", round(ap,3)
        )

        fold_results.append({
            "fold": fold,
            "model": name,
            "acc": acc,
            "auc": auc,
            "ap": ap,
            "loss": loss,
            "f1": f1,
        })

# ============================================================
# Final metrics
# ============================================================

# ============================================================
# Results
# ============================================================
# Two views of the same runs:
#   CV SUMMARY  - mean and std ACROSS folds, which exposes fold-to-fold variance
#                 (important here: with ~51 samples a fold holds only ~5 rows).
#   OOF SUMMARY - metrics over the pooled out-of-fold predictions, a single
#                 estimate over every sample.
#
# Report both. AUC on a 5-row fold is extremely noisy in isolation.
results_df = pd.DataFrame(fold_results)

print("\n=== CV SUMMARY ===")

print(
    results_df
    .groupby("model")[["acc","auc","ap","loss","f1"]]
    .agg(["mean","std"])
)

print("\n=== OOF SUMMARY ===")

for name, prob in [
    ("ESM2_MLP", mlp_probs_oof),
    ("GNN", gnn_probs_oof),
    ("STRUCT_MLP", struct_probs_oof),
    ("FIXED_ENSEMBLE", fixed_probs_oof),
    ("STACKED_ENSEMBLE", stacked_probs_oof),
]:
    pred = (prob >= 0.5).astype(int)

    print("\n", name)
    print("acc:", accuracy_score(y_valid, pred))
    print("auc:", roc_auc_score(y_valid, prob))
    print("pr_auc:", average_precision_score(y_valid, prob))
    print("loss:", log_loss(y_valid, prob, labels=[0,1]))
    print("f1:", f1_score(y_valid, pred, zero_division=0))

---

## Results, caveats and next steps

**No metrics are stored in this notebook** — the modeling cell above has no
saved output, so any accuracy or AUC figure would have to come from running it
yourself. Report both views it prints: the per-fold mean ± std (with ~51
samples a fold holds only ~5 rows, so single-fold AUC is extremely noisy) and
the pooled out-of-fold summary.

Known limitations, in rough order of how much they matter:

- **Sample size.** 51 variants from a single gene. Fold-to-fold variance will
  dominate, and nothing here should be read as generalising beyond APC.
- **Window assumption.** Centering on the first differing residue is correct
  for missense variants but structurally incomplete for frameshift and nonsense
  variants, where everything downstream changes.
- **Model selection.** `pick_best_cif` takes the lexicographically first model,
  not the most confident one. OpenFold3 emits per-model confidence (pLDDT/PAE)
  that is currently unused entirely.
- **GNN early stopping** selects on training loss, since the validation fold
  must stay unseen — it prevents divergence, not overfitting.
- **Smallest ESM2 checkpoint** (`esm2_t6_8M_UR50D`, 6 layers) was chosen to fit
  the compute budget.

Natural next steps: use the confidence scores as features (and to rank models),
fold several seeds per variant, extend past a single gene, and check whether
the learned stack actually beats the fixed blend — the single most informative
number this notebook can produce.